# 🎵 AI Cover — 一键 Notebook（Demucs → RVC → 混音）

> 适配 Windows / RTX40 系列。建议：分轨在 `aicover` 环境，RVC 推理用 `RVC1006Nvidia\runtime`，混音任意环境均可。

## 0. 配置

In [2]:
print("1")
print()
print('\n\n\n', end='\n')
print()
print("1")

1






1


In [1]:
from pathlib import Path

# === 必改项（按你的电脑路径改好再运行下一格）===
BASE     = Path(r"D:\change voice\ai-cover-project")              # 你的项目根目录
RVC_DIR  = Path(r"D:\5992\change voice\RVC1006Nvidia")            # RVC 仓库根目录
INPUT    = BASE / "input" / "source.wav"                          # 待处理歌曲（建议 WAV；不为 WAV 会自动转换）
MODEL_FN = "XXXTENTACION.pth"                                     # 放在 RVC 仓库 assets\weights\ 下的模型文件名
INDEX    = BASE / "vc_model" / "XXXTENTACION.index"               # 可选：索引文件绝对路径；不用索引可留空

# 推理参数
F0_METHOD   = "crepe"     # crepe / rmvpe（rmvpe 需 onnxruntime-gpu）
F0_KEY      = 0           # 半音上/下移
INDEX_RATE  = 0.0         # 0 关闭索引；0.78 打开索引（需 INDEX 存在 & faiss-cpu）

# === 不用改 ===
assert BASE.exists(), f"BASE 不存在：{BASE}"
assert RVC_DIR.exists(), f"RVC_DIR 不存在：{RVC_DIR}"
INPUT.parent.mkdir(parents=True, exist_ok=True)
print("BASE   :", BASE)
print("RVC_DIR:", RVC_DIR)
print("INPUT  :", INPUT, "(exists:", INPUT.exists(), ")")
print("MODEL_FN:", MODEL_FN)
print("INDEX  :", INDEX, "(exists:", INDEX.exists(), ")")

BASE   : D:\change voice\ai-cover-project
RVC_DIR: D:\5992\change voice\RVC1006Nvidia
INPUT  : D:\change voice\ai-cover-project\input\source.wav (exists: True )
MODEL_FN: XXXTENTACION.pth
INDEX  : D:\change voice\ai-cover-project\vc_model\XXXTENTACION.index (exists: True )


## 1. 环境检测（当前内核）

In [2]:
import sys, torch

print("Python :", sys.executable)
try:
    print("CUDA?   ", torch.cuda.is_available(), "CUDA", getattr(torch.version, "cuda", None))
    if torch.cuda.is_available():
        print("Device  :", torch.cuda.get_device_name(0))
except Exception as e:
    print("Torch not available or error:", e)

Python : C:\anaconda\envs\aicover\python.exe
CUDA?    True CUDA 12.1
Device  : NVIDIA GeForce RTX 4060 Laptop GPU


## 2. 分轨（Demucs）

In [ ]:
import sys, subprocess, librosa, soundfile as sf, numpy as np, os
from pathlib import Path

# 安装 demucs（若已安装会秒过）；Windows 上建议 PyPI 最新版
subprocess.run([sys.executable, "-m", "pip", "install", "-U", "demucs", "librosa", "soundfile"], check=False)

# 准备输入：若不是 wav 或含非 ASCII 路径，转换到 ASCII WAV
conv_dir = BASE / "input"
conv_dir.mkdir(parents=True, exist_ok=True)
work_src = Path(INPUT)

try:
    s = str(work_src)
    if work_src.suffix.lower() != ".wav" or any(ord(c)>127 for c in s):
        y, sr = librosa.load(s, sr=44100, mono=False)
        # 转立体声 (channels, n) → 写成 (n, channels)
        if y.ndim == 1:
            y = np.vstack([y, y])
        if y.shape[0] > y.shape[1]:
            y = y.T
        work_src = conv_dir / "source_stereo.wav"
        sf.write(str(work_src), y, 44100)
        print("Converted to:", work_src)
except Exception as e:
    print("Pre-conversion skipped:", repr(e))

# Demucs 运行（优先 GPU，失败回退 CPU，再回退小模型）
def run_demucs(device: str, model: str):
    out = BASE / "stems"
    cmd = [sys.executable, "-m", "demucs.separate",
           "-d", device, "--two-stems", "vocals",
           "-n", model, str(work_src), "-o", str(out)]
    print("RUN:", " ".join(cmd))
    p = subprocess.run(cmd, capture_output=True, text=True)
    print("=== STDOUT (tail) ===\n", p.stdout[-800:])
    print("=== STDERR (tail) ===\n", p.stderr[-800:])
    return p.returncode == 0

ok = run_demucs("cuda", "mdx_extra")
if not ok:
    print("[Info] CUDA 失败，试 CPU 同模型…")
    ok = run_demucs("cpu", "mdx_extra")
if not ok:
    print("[Info] 继续试小模型 mdx_q（CPU）…")
    # mdx_q 可能需要 diffq；装不上就跳过
    subprocess.run([sys.executable, "-m", "pip", "install", "-U", "diffq"], check=False)
    ok = run_demucs("cpu", "mdx_q")

assert ok, "Demucs 分轨失败，请检查上面 STDERR。"

# 记录人声/伴奏路径
# Demucs 输出结构：stems/<model>/<file_stem>/vocals.wav & no_vocals.wav
stems_root = BASE / "stems"
vocals_cands = sorted(stems_root.rglob("vocals.wav"), key=lambda p: p.stat().st_mtime)
assert vocals_cands, "找不到 vocals.wav"
vocals_path = vocals_cands[-1]
inst_path = vocals_path.parent / "no_vocals.wav"
if not inst_path.exists():
    for alt in ("accompaniment.wav", "instrumental.wav", "other.wav"):
        if (vocals_path.parent / alt).exists():
            inst_path = vocals_path.parent / alt
            break

print("vocals:", vocals_path)
print("inst  :", inst_path)

RUN: C:\anaconda\envs\aicover\python.exe -m demucs.separate -d cuda --two-stems vocals -n mdx_extra D:\change voice\ai-cover-project\input\source.wav -o D:\change voice\ai-cover-project\stems
=== STDOUT (tail) ===
 Selected model is a bag of 4 models. You will see that many progress bars per track.
Separated tracks will be stored in D:\change voice\ai-cover-project\stems\mdx_extra
Separating track D:\change voice\ai-cover-project\input\source.wav

=== STDERR (tail) ===
 Traceback (most recent call last):
  File "C:\anaconda\envs\aicover\lib\runpy.py", line 196, in _run_module_as_main
    return _run_code(code, main_globals, None,
  File "C:\anaconda\envs\aicover\lib\runpy.py", line 86, in _run_code
    exec(code, run_globals)
  File "C:\anaconda\envs\aicover\lib\site-packages\demucs\separate.py", line 219, in <module>
    main()
  File "C:\anaconda\envs\aicover\lib\site-packages\demucs\separate.py", line 171, in main
    wav -= ref.mean()
RuntimeError: unsupported operation: more than 

## 3. 准备 RVC runtime（CUDA 版 Torch + 依赖 + 补丁）

In [ ]:
import sys, subprocess, re, shutil
from pathlib import Path

RVC_PY = RVC_DIR / "runtime" / "python.exe"
assert RVC_PY.exists(), f"找不到 runtime python: {RVC_PY}"

def pip_run(args):
    return subprocess.run([str(RVC_PY), "-m", "pip", *args], text=True)

# 3.1 安装 CUDA 版 PyTorch（cu121）
pip_run(["uninstall", "-y", "torch", "torchvision", "torchaudio"])
pip_run(["cache", "purge"])
subprocess.check_call([str(RVC_PY), "-m", "pip", "install", "--no-cache-dir", "--force-reinstall",
                       "--index-url", "https://download.pytorch.org/whl/cu121",
                       "torch==2.3.1+cu121", "torchvision==0.18.1+cu121", "torchaudio==2.3.1+cu121"])

# 3.2 基础依赖（以及 Numpy/Scipy 组合，避免 SciPy ABI 报错）
subprocess.check_call([str(RVC_PY), "-m", "pip", "install", "--no-cache-dir", "--force-reinstall", "numpy==1.26.4"])
subprocess.check_call([str(RVC_PY), "-m", "pip", "install", "--no-cache-dir", "--only-binary=:all:", "scipy==1.10.1"])
pip_run(["install", "-U", "python-dotenv", "soundfile", "praat-parselmouth", "pyworld", "av"])

# 3.3 补丁 1：PyAV 的 rb/wb → r/w；或直接替换 load_audio 为 librosa/soundfile 版本
audio_py = RVC_DIR / "infer" / "lib" / "audio.py"
assert audio_py.exists(), audio_py
bak1 = audio_py.with_suffix(".py.bak2")
if not bak1.exists():
    shutil.copy2(audio_py, bak1)

src = audio_py.read_text(encoding="utf-8")
# 直接替换为无 PyAV 的 load_audio 实现，添加 audio2 stub
load_audio_impl = r'''
def load_audio(path, sr):
    import numpy as np
    import soundfile as sf
    import librosa
    try:
        y, _ = librosa.load(path, sr=sr, mono=True)
        return y.astype(np.float32)
    except Exception:
        data, fr = sf.read(path, always_2d=False)
        if data.ndim > 1:
            data = data.mean(axis=1)
        data = data.astype(np.float32)
        if fr != sr:
            data = librosa.resample(data, orig_sr=fr, target_sr=sr)
        return data.astype(np.float32)
def audio2(*args, **kwargs):
    raise NotImplementedError("audio2 disabled; load_audio handles decode+resample.")
'''
import re
src2 = re.sub(r'\ndef\s+load_audio\s*\(.*?\)\s*:[\s\S]*?(?=\ndef\s|\Z)', '\n'+load_audio_impl+'\n', src, flags=re.DOTALL)
if "def audio2(" in src2:
    src2 = re.sub(r'\ndef\s+audio2\s*\(.*?\)\s*:[\s\S]*?(?=\ndef\s|\Z)', '\n'+load_audio_impl.split("def audio2",1)[1]+'\n', src2, flags=re.DOTALL)
else:
    src2 += '\n' + load_audio_impl.split("def audio2",1)[1] + '\n'

if src2 != src:
    audio_py.write_text(src2, encoding="utf-8")
    print("[Patched] infer/lib/audio.py -> use librosa/soundfile")

# 3.4 补丁 2：index_path 为空时的 None 防护
modules_py = RVC_DIR / "infer" / "modules" / "vc" / "modules.py"
assert modules_py.exists(), modules_py
bak2 = modules_py.with_suffix(".py.bak_index")
if not bak2.exists():
    shutil.copy2(modules_py, bak2)

txt = modules_py.read_text(encoding="utf-8")
txt2 = re.sub(r'file_index\.strip\(" ?"\)', '(file_index or "").strip()', txt)
txt2 = re.sub(r'file_index2\.strip\(" ?"\)', '(file_index2 or "").strip()', txt2)
if txt2 != txt:
    modules_py.write_text(txt2, encoding="utf-8")
    print("[Patched] infer/modules/vc/modules.py -> index None guard")

# 3.5 自检 CUDA
chk = subprocess.run([str(RVC_PY), "-c", "import torch;print('cuda?',torch.cuda.is_available(),'cuda',torch.version.cuda)"],
                     capture_output=True, text=True)
print(chk.stdout or chk.stderr)

# 3.6 （可选）rmvpe 需要 onnxruntime-gpu
if F0_METHOD.lower() == "rmvpe":
    pip_run(["install", "-U", "onnxruntime-gpu"])

## 4. RVC 推理（CUDA）

In [ ]:
import os, subprocess
from pathlib import Path

PY      = RVC_DIR / "runtime" / "python.exe"
script  = RVC_DIR / "tools" / "infer_cli.py"

vc_dir = BASE / "vc_output"
vc_dir.mkdir(parents=True, exist_ok=True)
OUTPUT_VOX = vc_dir / (Path(vocals_path).stem + "_rvc.wav")

# 清理之前的 0 字节文件（若有）
if OUTPUT_VOX.exists() and OUTPUT_VOX.stat().st_size == 0:
    OUTPUT_VOX.unlink()

env = os.environ.copy()
env["PYTHONPATH"] = str(RVC_DIR) + os.pathsep + env.get("PYTHONPATH", "")

cmd = [str(PY), str(script),
       "--input_path", str(vocals_path),
       "--opt_path",   str(OUTPUT_VOX),
       "--model_name", MODEL_FN,
       "--index_rate", str(INDEX_RATE),
       "--f0method",   F0_METHOD,
       "--f0up_key",   str(F0_KEY),
       "--device",     "cuda"]

# 当 INDEX_RATE=0 时仍传空字符串可避免某些分支报错；启用索引则传绝对路径
cmd += ["--index_path", str(INDEX if INDEX.exists() and INDEX_RATE>0 else "")]

print("RUN:", " ".join(map(str, cmd)))
p = subprocess.run(cmd, cwd=str(RVC_DIR), env=env, capture_output=True, text=True)
print("=== STDOUT (tail) ===\n", p.stdout[-1200:])
print("=== STDERR (tail) ===\n", p.stderr[-1200:])
print("Return code:", p.returncode, "| Exists:", OUTPUT_VOX.exists(), "| Size:", (OUTPUT_VOX.stat().st_size if OUTPUT_VOX.exists() else 0))

# 若失败则回退使用原人声进入混音
if (not OUTPUT_VOX.exists()) or OUTPUT_VOX.stat().st_size < 2048:
    print("[Warn] RVC 输出无效，将回退使用原人声进入混音。")
    OUTPUT_VOX = vocals_path
else:
    print("[OK] 使用转换后人声：", OUTPUT_VOX)

## 5. 混音导出（Pedalboard）

In [ ]:
import sys, subprocess
# 确保当前内核安装了混音依赖（如果你要在 aicover 内核混音，这里会装到当前内核）
subprocess.run([sys.executable, "-m", "pip", "install", "-U", "pedalboard", "librosa", "soundfile"], check=False)

In [ ]:
import os, sys, librosa, soundfile as sf
import numpy as np
from pathlib import Path
from pedalboard import Pedalboard, HighpassFilter, Compressor, Gain, Reverb, Limiter
from pedalboard.io import AudioFile

# De-esser 兜底
try:
    from pedalboard import Deesser
    _deess = Deesser()
except Exception:
    _deess = None
    try:
        from pedalboard import PeakFilter
        _deess = PeakFilter(frequency_hz=7500.0, gain_db=-4.5, q=1.2)
    except Exception:
        try:
            from pedalboard import HighShelfFilter
            _deess = HighShelfFilter(cutoff_frequency_hz=7000.0, gain_db=-4.0)
        except Exception:
            _deess = Gain(gain_db=0.0)

# 选择人声（优先 RVC 输出，检测非空）
def nonempty(p: Path, min_bytes=2048):
    try:
        return p.exists() and p.stat().st_size >= min_bytes
    except Exception:
        return False

vox_in  = OUTPUT_VOX if nonempty(OUTPUT_VOX) else vocals_path
inst_in = Path(inst_path)

print("[Files]")
print("  Vocal in :", vox_in, vox_in.exists(), (vox_in.stat().st_size if vox_in.exists() else 0))
print("  Inst  in :", inst_in, inst_in.exists(), (inst_in.stat().st_size if inst_in.exists() else 0))

TARGET_SR = 48000
inst_np, _ = librosa.load(str(inst_in), sr=TARGET_SR, mono=False)
vox_np,  _ = librosa.load(str(vox_in),  sr=TARGET_SR, mono=False)

def to_ch_first(y: np.ndarray) -> np.ndarray:
    y = np.asarray(y)
    if y.ndim == 1:
        return np.vstack([y, y])
    return y.T if y.shape[0] > y.shape[1] else y

inst = to_ch_first(inst_np)
vox  = to_ch_first(vox_np)

L = min(inst.shape[-1], vox.shape[-1])
inst = inst[:, :L]
vox  = vox[:,  :L]

def rms(x):
    return np.sqrt(np.mean(np.square(x), axis=-1, keepdims=True) + 1e-12)

inst_rms = float(np.mean(rms(inst)))
vox_rms  = float(np.mean(rms(vox)))
target_rms = max(inst_rms, 1e-4)
if vox_rms > 0:
    vox = vox * (target_rms / vox_rms * 0.9)

chain = Pedalboard([
    HighpassFilter(cutoff_frequency_hz=80.0),
    _deess,
    Compressor(threshold_db=-18, ratio=2.5),
    Reverb(room_size=0.12, wet_level=0.08, dry_level=0.92),
    Gain(gain_db=0.0),
])

vox_proc = np.vstack([chain(vox[ch], TARGET_SR) for ch in range(vox.shape[0])])
mix = inst + 0.9 * vox_proc

limit = Pedalboard([Limiter(threshold_db=-1.0)])
mix = np.vstack([limit(mix[ch], TARGET_SR) for ch in range(mix.shape[0])])

mx = float(np.max(np.abs(mix)))
if mx > 0.999:
    mix = mix / mx * 0.999

final_dir = BASE / "mix"
final_dir.mkdir(parents=True, exist_ok=True)
final_wav = final_dir / "final_mix.wav"

with AudioFile(str(final_wav), 'w', TARGET_SR, mix.shape[0]) as f:
    f.write(mix.T.astype(np.float32))

print("✅ Done:", final_wav, "| size:", final_wav.stat().st_size)
str(final_wav)

## 6. 可选：响度归一化到 -14 LUFS

In [ ]:
# 需要 pyloudnorm；若不需要可跳过
import sys, subprocess; subprocess.run([sys.executable, "-m", "pip", "install", "-U", "pyloudnorm"], check=False)

import numpy as np, soundfile as sf, pyloudnorm as pyln
from pathlib import Path

target_lufs = -14.0
in_wav  = (BASE / "mix" / "final_mix.wav")
out_wav = (BASE / "mix" / "final_mix_lufs-14.wav")

y, sr = sf.read(str(in_wav), always_2d=False)
if y.ndim > 1:
    y_mono = y.mean(axis=1)
else:
    y_mono = y
meter = pyln.Meter(sr)
lufs = meter.integrated_loudness(y_mono)
gain_db = float(target_lufs - lufs)
amp = 10 ** (gain_db / 20.0)
y_out = (y * amp).astype(np.float32)
sf.write(str(out_wav), y_out, sr)
print(f"原始: {lufs:.2f} LUFS → 目标: {target_lufs} LUFS，增益 {gain_db:.2f} dB")
print("写出：", out_wav)
str(out_wav)